In [ ]:
# Install the GitHub-pinned native library and profile registry.
from pathlib import Path
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory
import json
import os

SETUP_COMMIT = "065f06732812e6176935806520e1b50197140bf7"
setup_url = (
    "https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/"
    f"{SETUP_COMMIT}/kaggle/cube555_github/setup.py"
)
with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve(setup_url, setup_dir + "/setup.py")
    run_path(setup_file)

SOLVER_COMMIT = os.environ["MULTIGPUBEAMSEARCH_SOURCE_COMMIT"]
WORK_ROOT = Path(
    os.environ.get(
        "MULTIGPUBEAMSEARCH_HOME",
        Path.home() / ".cache/multigpubeamsearch/cube555",
    )
)
source_notebook = (
    WORK_ROOT
    / "sources"
    / SOLVER_COMMIT
    / "kaggle/cube555_github/cube555-native-2xt4-blend.ipynb"
)
NOTEBOOK_PATH = WORK_ROOT / "publication-notebook.ipynb"
notebook = json.loads(source_notebook.read_text(encoding="utf-8"))
notebook["cells"][0]["source"] = [
    line.replace("__SETUP_COMMIT__", SETUP_COMMIT)
    for line in notebook["cells"][0]["source"]
]
NOTEBOOK_PATH.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")


In [ ]:
import cayleypy as cp
import multigpubeamsearch as beam
import os
from pathlib import Path

# Define your CayleyPy graph and predictor here, or use the bundled Cube555 example.
# PREDICTOR may be a CayleyPy Predictor, supported torch model, NativeModel, or NativeEnsemble.
GRAPH = None
PREDICTOR = None
START_STATE = None
START_STATES = None
# Example:
# GRAPH = cp.CayleyGraph(
#     cp.CayleyGraphDef.create(
#         generators=[[1, 0]],
#         generator_names=["swap"],
#         central_state=[0, 1],
#     ),
#     device="cpu",
# )
# PREDICTOR = your_cayleypy_predictor

# Inputs for file-based graphs and start states.
INPUT_BUNDLE_URL = "https://github.com/TryDotAtwo/MultiGPUBeamSearch/releases/download/cube555-inputs-20261001/cube555-inputs.zip"
INPUT_CACHE_DIR = None
GRAPH_INFO_PATH = None
STATE_CSV_PATH = None
COMPETITION_ROOT = None
STATE_ID_COLUMN = "initial_state_id"
STATE_COLUMN = "initial_state"
STATE_SEPARATOR = ","

# Select CSV rows, or set PUZZLE_IDS when using START_STATES directly.
PUZZLE_IDS = None
PUZZLE_ID_START = 1020
PUZZLE_ID_END = 1020
INDICES_URL = None
RANDOM_PUZZLE = False
START_STATE_OVERRIDE = None

# Search controls.
BEAM_WIDTH = 1_048_576
MAX_STEPS = 140
BEAM_MODE = "simple"
USE_CHILD_SCORES = True
RETURN_PATH = True
NUM_GPUS = None
SOLUTION_MODE = "collect"
COLLECT_EXTRA_DEPTHS = 1
MAX_COLLECTED_SOLUTIONS = 2_000
TOUCH_BFS_RADIUS = 5
TOUCH_BFS_MAX_ENTRIES = 30_000_000
REFLECT_MODE = "off"
REFLECT_SOURCE_CSV = None
REFLECT_ID_COLUMN = "puzzle_id"
REFLECT_PATH_COLUMN = "solution"
REFLECTION_PATHS = None

# Model selection. Set PREDICTOR directly, or provide a factory with this signature:
# factory(graph, model_root, graph_info_path) -> CayleyPy predictor / NativeModel / NativeEnsemble.
MODEL_FACTORY = "cube555_blend"
MODEL_ROOT = None
CHECKPOINT_PATH = "q555_f1_bell2k.pt"
RESMLP_CHECKPOINT_PATH = "q555_2k_BEST.pt"
LAYOUT_PATH = "piece_layout_555.json"
TRANSFORMER_WEIGHT = 0.8

# Results and optional CayleyPy results-catalog publication.
OUTPUT_ROOT = "beam_results"
PUBLISH_RESULTS = True
PUBLISH_MODEL_METADATA = None
AUTHOR = "trydotatwo"
KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = int(os.environ.get("KAGGLE_KERNEL_VERSION", "180"))
COMPETITION = None
PUZZLE_TYPE = None
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_KERNEL_ID"))


# CayleyPy + native multi-GPU beam search

The search uses CayleyPy's graph and beam-search APIs. For another problem, set GRAPH, PREDICTOR, and the start state in the first cell; the search, collect/BFS/reflection, replay, profile selection, and local saving stay the same.

A native predictor needs a registered exporter. Pass your own NativeModel or supported predictor directly. MODEL_FACTORY=None selects CaylePy's default Hamming predictor.

## Load the graph, states, and model

The Cube555 model builder below is only the notebook's default example. Replace GRAPH and PREDICTOR in the first cell, or set MODEL_FACTORY to a function that returns your model. The beam-search code does not depend on the graph or model family.

In [ ]:
# Use supplied CaylePy objects and load only inputs that are still missing.
needs_graph_file = GRAPH is None and GRAPH_INFO_PATH is None
needs_state_file = START_STATE is None and START_STATES is None and STATE_CSV_PATH is None
needs_cube_model = (
    PREDICTOR is None
    and MODEL_FACTORY == "cube555_blend"
    and (MODEL_ROOT is None or GRAPH_INFO_PATH is None)
)
input_root = (
    beam.download_bundle(INPUT_BUNDLE_URL, INPUT_CACHE_DIR)
    if needs_graph_file or needs_state_file or needs_cube_model
    else None
)

model_root = Path(MODEL_ROOT) if MODEL_ROOT is not None else (
    input_root / "model" if input_root is not None else None
)
graph_info_path = Path(GRAPH_INFO_PATH) if GRAPH_INFO_PATH is not None else (
    model_root / "puzzle_info.json" if model_root is not None else None
)
competition_root = (
    Path(COMPETITION_ROOT)
    if COMPETITION_ROOT is not None
    else (input_root / "competition" if input_root is not None else None)
)
state_csv_path = Path(STATE_CSV_PATH) if STATE_CSV_PATH is not None else (
    competition_root / "test.csv" if competition_root is not None else None
)

if MODEL_FACTORY == "cube555_blend" and PREDICTOR is None and GRAPH is not None and GRAPH_INFO_PATH is None:
    raise ValueError("The Cube555 example model needs GRAPH_INFO_PATH. Set PREDICTOR or a model factory for a custom graph.")

if GRAPH is not None:
    graph = GRAPH
elif graph_info_path is not None:
    graph = beam.load_graph(graph_info_path, device="cpu", random_seed=42)
else:
    raise ValueError("Set GRAPH or GRAPH_INFO_PATH in the first cell.")

if START_STATE is not None:
    start_states = {"custom": START_STATE}
    puzzle_ids = ["custom"]
elif START_STATES is not None:
    if not isinstance(START_STATES, dict) or not START_STATES:
        raise ValueError("START_STATES must be a nonempty mapping of puzzle IDs to states.")
    start_states = {str(key): value for key, value in START_STATES.items()}
    puzzle_ids = [str(value) for value in PUZZLE_IDS] if PUZZLE_IDS is not None else list(start_states)
else:
    if state_csv_path is None:
        raise ValueError("Set START_STATE, START_STATES, or STATE_CSV_PATH in the first cell.")
    start_states = beam.read_states(
        state_csv_path,
        id_column=STATE_ID_COLUMN,
        state_column=STATE_COLUMN,
        separator=STATE_SEPARATOR,
    )
    puzzle_ids = (
        [str(value) for value in PUZZLE_IDS]
        if PUZZLE_IDS is not None
        else beam.select_ids(
            PUZZLE_ID_START,
            PUZZLE_ID_END,
            indices_url=INDICES_URL,
            random_choice=RANDOM_PUZZLE,
        )
    )

def build_cube555_blend(graph, model_root, graph_info_path):
    """Build the notebook's default example; replace this with your model factory."""
    import sys
    import tempfile
    from decimal import Decimal
    from multigpubeamsearch.contracts import GraphContract

    if model_root is None or graph_info_path is None:
        raise ValueError("The Cube555 example needs its model files and graph definition.")
    source_root = Path(beam.__file__).resolve().parent / "_native_source"
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))
    from tools.cube555.export import export_native_models

    def model_file(value):
        candidate = Path(value)
        return candidate if candidate.is_absolute() else model_root / candidate

    export_root = Path(INPUT_CACHE_DIR or Path.home() / ".cache/multigpubeamsearch") / "exports"
    export_root.mkdir(parents=True, exist_ok=True)
    output = Path(tempfile.mkdtemp(prefix="blend-", dir=export_root)) / "models"
    members = export_native_models(
        model_file(CHECKPOINT_PATH),
        model_file(RESMLP_CHECKPOINT_PATH),
        model_file(LAYOUT_PATH),
        graph_info_path,
        output,
        contract=GraphContract.from_graph(graph, graph.definition.central_state),
    )
    transformer_weight = float(TRANSFORMER_WEIGHT)
    mlp_weight = float(Decimal(1) - Decimal(str(TRANSFORMER_WEIGHT)))
    return beam.NativeEnsemble(members, (transformer_weight, mlp_weight))

if PREDICTOR is not None:
    predictor = PREDICTOR
elif MODEL_FACTORY is None:
    predictor = None
elif MODEL_FACTORY == "cube555_blend":
    predictor = build_cube555_blend(graph, model_root, graph_info_path)
elif callable(MODEL_FACTORY):
    predictor = MODEL_FACTORY(graph, model_root, graph_info_path)
else:
    raise ValueError("MODEL_FACTORY must be None, 'cube555_blend', or a model factory function.")

if REFLECTION_PATHS is not None:
    reflection_paths = {str(key): value for key, value in REFLECTION_PATHS.items()}
elif REFLECT_SOURCE_CSV:
    reflection_paths = beam.read_paths(
        REFLECT_SOURCE_CSV, graph, id_column=REFLECT_ID_COLUMN, path_column=REFLECT_PATH_COLUMN
    )
else:
    reflection_paths = {}

beam.enable_native(beam.NativeOptions(num_gpus=NUM_GPUS))

# Catalog identity is required for generic graphs. The bundled default fills
# its own known values; custom graphs are never mislabeled as Cube555.
catalog_competition = COMPETITION
catalog_puzzle_type = PUZZLE_TYPE
if GRAPH is None and GRAPH_INFO_PATH is None:
    catalog_competition = catalog_competition or "cayley-py-555-cube"
    catalog_puzzle_type = catalog_puzzle_type or "cube_5/5/5"


## Search and save

The same CayleyPy call runs for every graph and predictor. Results are replay-checked before local save; publication uses the generic graph definition and requires a catalog competition and puzzle type.


In [ ]:
# Keep models, workers, and GPU buffers alive across selected start states.
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
publication_reports = []

with beam.NativeSession():
    for puzzle_id in puzzle_ids:
        start_state = (
            START_STATE_OVERRIDE
            if START_STATE_OVERRIDE is not None
            else start_states[puzzle_id]
        )
        result = graph.beam_search(
            start_state=start_state,
            predictor=predictor,
            backend="native",
            beam_width=BEAM_WIDTH,
            max_steps=MAX_STEPS,
            beam_mode=BEAM_MODE,
            use_child_scores=USE_CHILD_SCORES,
            return_path=RETURN_PATH,
            solution_mode=SOLUTION_MODE,
            collect_extra_depths=COLLECT_EXTRA_DEPTHS,
            max_collected_solutions=MAX_COLLECTED_SOLUTIONS,
            bfs_radius=TOUCH_BFS_RADIUS,
            bfs_max_entries=TOUCH_BFS_MAX_ENTRIES,
            reflect_mode=REFLECT_MODE,
            reflection_source=reflection_paths.get(str(puzzle_id)),
        )

        saved = beam.save_result(
            result,
            graph,
            start_state,
            OUTPUT_ROOT,
            metadata={"problem_id": str(puzzle_id)},
        )

        publication = None
        if PUBLISH_RESULTS and result.solutions:
            if not IS_KAGGLE:
                publication = {"status": "local_only"}
                print("Saved locally. Catalog publication requires a Kaggle-provenance run.")
            elif len(result.native_metadata.get("devices", ())) != 2:
                publication = {"status": "local_only"}
                print("Saved locally. The current results catalog accepts two-GPU runs.")
            elif not catalog_competition or not catalog_puzzle_type:
                publication = {"status": "catalog_identity_required"}
                print("Saved locally. Set COMPETITION and PUZZLE_TYPE to publish this graph.")
            else:
                try:
                    publication = beam.publish_result(
                        result,
                        graph,
                        start_state,
                        puzzle_id=puzzle_id,
                        predictor=predictor,
                        author=AUTHOR,
                        kaggle_owner=KAGGLE_OWNER,
                        kaggle_slug=KAGGLE_SLUG,
                        kaggle_version=KAGGLE_VERSION,
                        notebook_path=NOTEBOOK_PATH,
                        solver_commit=SOLVER_COMMIT,
                        competition=catalog_competition,
                        puzzle_type=catalog_puzzle_type,
                        model_metadata=PUBLISH_MODEL_METADATA,
                        directory=OUTPUT_ROOT,
                    )
                except ValueError as error:
                    publication = {"status": "not_published", "reason": str(error)}
                    print(f"Saved locally. Catalog rejected this model/problem metadata: {error}")

        if publication is not None:
            publication_reports.append({"problem_id": str(puzzle_id), **publication})

        print(
            f"Puzzle {puzzle_id}: best length {result.path_length}; "
            f"solutions {len(result.solutions)}; saved {saved}; "
            f"publication {None if publication is None else publication['status']}"
        )

if publication_reports:
    status_path = Path(OUTPUT_ROOT) / "publication_status.json"
    status_path.write_text(json.dumps(publication_reports, indent=2), encoding="utf-8")
    print(f"Publication receipts: {status_path}")
